# Lab 1: Intro and Data Cleaning
**Author:** Aditya Narayan Ray


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

print("Aditya Narayan Ray")


In [ ]:
# Load dataset and replace any hidden empty spaces with actual Null (NaN)
df = pd.read_csv("hotel_booking.csv")
df = df.replace(r'^\s*$', np.nan, regex=True)

# 1. Standard ML Notebook Data Inspection
print("--- First 5 Rows ---")
display(df.head())

print("\n--- Dataset Info ---")
display(df.info())

print("\n--- Statistical Summary ---")
display(df.describe())

# 2. Display ALL rows that contain at least one missing value
print("\n--- Rows with Missing Values ---")
missing_value_rows = df[df.isnull().any(axis=1)]
display(missing_value_rows)


In [ ]:
# 3. Histogram of Missing Values
print("\n--- Histogram of Missing Values ---")
missing_counts = df.isnull().sum()
missing_counts = missing_counts[missing_counts > 0] # Filter to only show columns with missing values

if not missing_counts.empty:
    plt.figure(figsize=(10, 6))
    # Using plt.bar directly prevents the empty axes issue
    plt.bar(missing_counts.index, missing_counts.values, color='salmon')
    plt.title('Count of Missing Values per Column (Before Cleaning)', fontsize=14)
    plt.xlabel('Columns', fontsize=12)
    plt.ylabel('Number of Missing Values', fontsize=12)
    plt.xticks(rotation=45) # Rotated to 45 degrees so column names don't overlap
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found.")


In [ ]:
# Handle Missing Values
# Drop 'company' entirely because 94% of its data is missing
df_clean = df.drop(columns=['company'])

# Fill missing 'agent' IDs with 0
df_clean['agent'] = df_clean['agent'].fillna(0)

# Fill missing 'country' values with the statistical mode
df_clean['country'] = df_clean['country'].fillna(df_clean['country'].mode()[0])

# Fill the missing 'children' values with 0
df_clean['children'] = df_clean['children'].fillna(0)

# Standardize Data Types
df_clean['children'] = df_clean['children'].astype(int)
df_clean['agent'] = df_clean['agent'].astype(int)

# Remove Logical Errors (Drop rows where total guests equal 0)
guest_filter = (df_clean['adults'] + df_clean['children'] + df_clean['babies']) > 0
df_clean = df_clean[guest_filter]

# Feature Engineering (Map month to numeric to include it in the correlation matrix)
month_map = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June': 6,
    'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12
}
df_clean['arrival_date_month_num'] = df_clean['arrival_date_month'].map(month_map)


In [ ]:
# 1. Verify Cleaned Data Inspection
print("--- First 5 Rows (Cleaned) ---")
display(df_clean.head())

# Verify no missing values remain
print("\n--- Remaining Missing Values ---")
print(df_clean.isnull().sum().max() == 0) # Should print True


In [ ]:
# 2. AFTER VISUALIZATIONS
# Histograms of Numerical Columns (Notice updated distributions)
df_clean.hist(bins=30, figsize=(20, 15), color='lightgreen', edgecolor='black')
plt.suptitle('After: Numerical Data Distributions', fontsize=16)
plt.show()

# Correlation Heatmap (Now includes arrival_date_month_num and updated records)
plt.figure(figsize=(14, 10))
numeric_df_after = df_clean.select_dtypes(include=[np.number])
sns.heatmap(numeric_df_after.corr(), annot=False, cmap='coolwarm', linewidths=0.5)
plt.title('After: Correlation Matrix')
plt.show()
